# Karakter Seviyesi Dil Modeli\nWikipedia Türkçe metinleriyle karakter seviyesinde Bigram ve LSTM dil modeli.\nModel bir sonraki karakteri tahmin etmeyi öğrenir ve yeni metin üretebilir.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim\nimport numpy as np, matplotlib.pyplot as plt, random, time\n\nSEED = 42\nrandom.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Cihaz: {device}')

## 1. Wikipedia'dan Veri Çekme

In [ ]:
import wikipediaapi\n\nwiki = wikipediaapi.Wikipedia(language='tr', user_agent='SLM-Egitim/1.0')\n\nmakaleler = ['Yapay zeka', 'Türkiye', 'İstanbul', 'Bilgisayar', 'Matematik', 'Fizik', 'Bilim', 'Tarih']\nham_metin = []\nfor baslik in makaleler:\n    sayfa = wiki.page(baslik)\n    if sayfa.exists():\n        ham_metin.append(sayfa.text)\n        print(f'  {baslik}: {len(sayfa.text):,} karakter')\n\nmetin = '\n'.join(ham_metin[:300000])\nprint(f'\nToplam: {len(metin):,} karakter')

## 2. Karakter Seviyesi Tokenization

In [ ]:
karakterler = sorted(set(metin))\nc2i = {c: i for i, c in enumerate(karakterler)}\ni2c = {i: c for c, i in c2i.items()}\nprint(f'Benzersiz karakter: {len(karakterler)}')\nprint(f'İlk 50: {" ".join(karakterler[:50])}')

In [ ]:
data = torch.tensor([c2i[c] for c in metin[:500000]], dtype=torch.long)\nn = int(0.9 * len(data))\ntrain_data, val_data = data[:n], data[n:]\nprint(f'Eğitim: {len(train_data):,}, Doğrulama: {len(val_data):,}')

## 3. Bigram Baseline Model

In [ ]:
vocab = len(karakterler)\nbigram = torch.zeros((vocab, vocab), dtype=torch.long)\nfor a, b in zip(train_data.tolist()[:-1], train_data.tolist()[1:]):\n    bigram[a][b] += 1\n\n# Görselleştir\nplt.figure(figsize=(8, 6))\nsubset = bigram[:40, :40].float()\nsubset[subset == 0] = 1\nplt.imshow(torch.log(subset), cmap='Blues')\nplt.colorbar(label='log frekans')\nplt.title('Bigram Matrisi (ilk 40 karakter)')\nplt.show()

In [ ]:
# Bigram ile metin üretimi\nstart = random.randint(0, len(train_data)-1)\nidx = train_data[start].item()\nuret = [i2c[idx]]\nfor _ in range(200):\n    probs = bigram[idx].float()\n    probs /= probs.sum()\n    idx = torch.multinomial(probs, 1).item()\n    uret.append(i2c[idx])\nprint('Bigram üretimi:')\nprint(''.join(uret))

## 4. LSTM Dil Modeli

In [ ]:
class LSTMSLM(nn.Module):\n    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, n_layers=2):\n        super().__init__()\n        self.embed = nn.Embedding(vocab_size, embed_dim)\n        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True, dropout=0.2)\n        self.fc = nn.Linear(hidden_dim, vocab_size)\n    def forward(self, x, hidden=None):\n        x = self.embed(x)\n        out, hidden = self.lstm(x, hidden)\n        return self.fc(out), hidden\n    def init_hidden(self, batch_size=1):\n        return (torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(device),\n                torch.zeros(self.lstm.num_layers, batch_size, self.lstm.hidden_size).to(device))\n\nmodel = LSTMSLM(vocab).to(device)\nprint(f'Parametre: {sum(p.numel() for p in model.parameters()):,}')

## 5. Eğitim

In [ ]:
def get_batch(data, batch_size, seq_len):\n    idx = torch.randint(0, len(data) - seq_len - 1, (batch_size,))\n    x = torch.stack([data[i:i+seq_len] for i in idx])\n    y = torch.stack([data[i+1:i+seq_len+1] for i in idx])\n    return x.to(device), y.to(device)\n\ncriterion = nn.CrossEntropyLoss()\noptimizer = optim.Adam(model.parameters(), lr=0.002)\nEPOCHS, SEQ_LEN, BATCH = 20, 64, 32\nlosses = []\n\nfor epoch in range(EPOCHS):\n    model.train()\n    x, y = get_batch(train_data, BATCH, SEQ_LEN)\n    hidden = model.init_hidden(BATCH)\n    for param in hidden: param.detach_()\n    \n    optimizer.zero_grad()\n    out, _ = model(x, hidden)\n    loss = criterion(out.view(-1, vocab), y.view(-1))\n    loss.backward()\n    torch.nn.utils.clip_grad_norm_(model.parameters(), 5)\n    optimizer.step()\n    losses.append(loss.item())\n    \n    if (epoch+1) % 5 == 0:\n        model.eval()\n        with torch.no_grad():\n            xv, yv = get_batch(val_data, BATCH, SEQ_LEN)\n            out, _ = model(xv, model.init_hidden(BATCH))\n            val_loss = criterion(out.view(-1, vocab_len), yv.view(-1)).item()\n        print(f'Epoch {epoch+1:3d}/{EPOCHS} | Train: {loss.item():.4f} | Val: {val_loss:.4f}')\n\nplt.plot(losses); plt.title('Eğitim Loss'); plt.xlabel('Adım'); plt.show()

## 6. Metin Üretimi

In [ ]:
model.eval()\nstart_char = 'Yapay zeka'\nidx = torch.tensor([[c2i[c] for c in start_char]], device=device)\nhidden = model.init_hidden(1)\nuret = list(start_char)\n\nwith torch.no_grad():\n    for _ in range(300):\n        out, hidden = model(idx, hidden)\n        probs = torch.softmax(out[0, -1] / 0.8, dim=-1)\n        next_idx = torch.multinomial(probs, 1).item()\n        uret.append(i2c[next_idx])\n        idx = torch.tensor([[next_idx]], device=device)\n\nprint(f'Prompt: "{start_char}"')\nprint(f'Üretilen: {''.join(uret)}')